[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VectorInstitute/synthetic-data-bootcamp/blob/main/implementations/qa_text_generation/01_baseline_evaluation.ipynb)

# Step 1 — Baseline Evaluation

Build a held-out **policy-document test set** and measure how a small instruction-tuned model performs before synthetic-data alignment.

## Learning objectives
- Ingest two finance policy documents (policy-dense + scope-boundary)
- Split paragraphs into **test** vs **train** sets
- Generate hard test Q&A with a teacher LLM
- Run baseline inference and LLM-as-judge scoring by failure mode

![Model Improvement Loop Flowchart](./images/model_improvement_loop_flowchart.png)


## Setup

In [1]:
import os
from pathlib import Path

from aieng.syn_data.text import (
    BASELINE_PREDICTIONS_PATH,
    BASELINE_SCORES_PATH,
    DEFAULT_TEST_PARAS_PER_DOC,
    PARAGRAPHS_PATH,
    TEST_SET_PATH,
    ParagraphSplit,
    QASample,
    build_paragraph_splits,
    create_judge_client,
    create_small_model_client,
    create_teacher_client,
    generate_test_qa_batch,
    list_domain_documents,
    load_implementation_dotenv,
    run_inference,
    save_baseline_results,
    save_typed_jsonl,
    score_predictions,
)
from rich.console import Console
from rich.panel import Panel
from rich.syntax import Syntax
from rich.table import Table


# Setting the notebook directory to the project's root folder
if Path("").absolute().name == "synthetic-data-bootcamp":
    print(f"Notebook path is already the root path: {Path('').absolute()}")
else:
    os.chdir(Path("").absolute().parent.parent)
    print(f"The notebook path has been set to: {Path('').absolute()}")

load_implementation_dotenv()

console = Console(width=100)

The notebook path has been set to: /Users/royajavadi/projects/synthetic-data-bootcamp


In [ ]:
import logging


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    force=True,  # override any earlier basicConfig from other libs
)

## 1. Load finance policy documents

We use two document archetypes:
- **Policy-dense** (CFPB credit card agreement) → format + vocabulary + multi-constraint
- **Scope-boundary** (SEC investor bulletin) → refusal calibration

Two document types = two different skills the small model needs to learn.

Policy-dense (CFPB credit card agreement)

Lots of rules, numbers, fees, defined terms
Tests: format compliance (answer as JSON/table), domain vocabulary (APR, grace period), multi-constraint questions (“what’s the fee and when is it charged?”)
Scope-boundary (SEC investor bulletin)

Explains what the document covers — and what it doesn’t
Tests: refusal calibration — answer in-scope questions, politely refuse out-of-scope ones (e.g. “Should I buy this stock?”)

So the bootcamp uses two archetypes to build a test set and training data that stress different weaknesses — closer to real deployments where models handle both “answer precisely from policy” and “know their limits.”

In code, ``failure_modes_for_paragraph()`` maps each role to the failure modes it’s meant to target.

In [6]:
specs = list_domain_documents("finance")
from rich.console import Console


console = Console()
table = Table(show_header=True, header_style="bold magenta")
table.add_column("doc_id")
table.add_column("title")
table.add_column("role")
table.add_column("domain")
table.add_column("source_url")
table.add_column("local_path")

for spec in specs:
    table.add_row(
        getattr(spec, "doc_id", ""),
        getattr(spec, "title", ""),
        str(getattr(spec, "role", "")),
        getattr(spec, "domain", ""),
        getattr(spec, "source_url", ""),
        getattr(spec, "local_path", ""),
    )

console.print(table)

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃ doc_id              ┃ title               ┃ role           ┃ domain  ┃ source_url         ┃ local_path          ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ cfpb_credit_card_a… │ CFPB Sample Credit  │ policy_dense   │ finance │ https://files.con… │ /home/coder/synthe… │
│                     │ Card Agreement      │                │         │                    │                     │
│ sec_investor_bulle… │ SEC Investor        │ scope_boundary │ finance │ https://www.sec.g… │ /home/coder/synthe… │
│                     │ Bulletin            │                │         │                    │                     │
└─────────────────────┴─────────────────────┴────────────────┴─────────┴────────────────────┴─────────────────────┘

## 2. Chunk into paragraphs and hold out test paragraphs

Randomly sample a few paragraphs per document for evaluation. **Never** use these paragraphs in Step 4 training.

In [7]:
paragraphs = build_paragraph_splits(
    "finance",
    n_test_per_doc=DEFAULT_TEST_PARAS_PER_DOC,
    seed=42,
)
test_paragraphs = [p for p in paragraphs if p.split == ParagraphSplit.TEST]
train_paragraphs = [p for p in paragraphs if p.split == ParagraphSplit.TRAIN]

console.print(f"[bold green]Total paragraphs:[/bold green] [cyan]{len(paragraphs)}[/cyan]")
console.print(
    f"[bold green]Test holdout:[/bold green] [cyan]{len(test_paragraphs)}[/cyan] | [bold green]Train reserve:[/bold green] [cyan]{len(train_paragraphs)}[/cyan]"
)

save_typed_jsonl(
    PARAGRAPHS_PATH,
    paragraphs,
    to_dict=lambda paragraph: paragraph.to_dict(),
)
PARAGRAPHS_PATH

Total paragraphs: 56

Test holdout: 20 | Train reserve: 36

PosixPath('/home/coder/synthetic-data-bootcamp/implementations/qa_text_generation/data/paragraphs.jsonl')

## 3. Generate hard test Q&A with the teacher model

Target the four small-model failure modes:
1. Format non-compliance
2. Domain vocabulary drift
3. Refusal vs engagement calibration
4. Multi-constraint collapse

In [8]:
teacher = create_teacher_client()

console.print(f"[bold magenta]Teacher LLM Base URL:[/bold magenta] [cyan]{teacher.settings.base_url}[/cyan]")

test_samples = generate_test_qa_batch(
    teacher,
    test_paragraphs,
    questions_per_para=3,
)
console.print(f"[bold green]Generated:[/bold green] [cyan]{len(test_samples)}[/cyan] test Q&A items")

save_typed_jsonl(
    TEST_SET_PATH,
    test_samples,
    to_dict=QASample.to_dict,
)

Teacher LLM Base URL: https://proxy.vectorinstitute.ai/v1

2026-07-28 21:18:36,626 DEBUG urllib3.connectionpool: Starting new HTTPS connection (1): proxy.vectorinstitute.ai:443


2026-07-28 21:18:38,572 DEBUG urllib3.connectionpool: https://proxy.vectorinstitute.ai:443 "POST /v1/chat/completions HTTP/1.1" 200 None
2026-07-28 21:18:38,574 DEBUG aieng.syn_data.text.clients: *********** Extracted JSON payload: *********** 
{
  "topics": [
    "Finance charge grace periods for new purchases",
    "Accrual dates for cash advance finance charges",
    "Calculation method for average daily balance of purchases",
    "Calculation method for average daily balance of cash advances",
    "Treatment of balance transfers relative to cash advances"
  ]
}
*********** End of JSON payload ***********
2026-07-28 21:18:38,577 DEBUG urllib3.connectionpool: Starting new HTTPS connection (1): proxy.vectorinstitute.ai:443
2026-07-28 21:18:40,471 DEBUG urllib3.connectionpool: https://proxy.vectorinstitute.ai:443 "POST /v1/chat/completions HTTP/1.1" 200 None
2026-07-28 21:18:40,474 DEBUG aieng.syn_data.text.clients: *********** Extracted JSON payload: *********** 
{
  "question": "Acco

Generated: 60 test Q&A items

PosixPath('/home/coder/synthetic-data-bootcamp/implementations/qa_text_generation/data/test/test_set.jsonl')

Altrnatively, you may already saved the generated tests. So you can continue with reading them without generation:

In [ ]:
from aieng.syn_data.text.io import load_typed_jsonl


test_samples = load_typed_jsonl(TEST_SET_PATH, QASample.from_dict)

In [ ]:
def show_qa_samples(samples):
    """Display a table of Q&A samples with IDs, questions, answers, and metadata.

    Parameters
    ----------
    samples : list
        A list of QASample objects or similar, each with id, question, gold_answer,
        failure_mode, and role attributes.
    """
    table = Table(title="Test Q&A Samples", show_lines=True)
    table.add_column("ID", style="cyan", no_wrap=True)
    table.add_column("Question", style="magenta")
    table.add_column("Answer", style="green")
    table.add_column("Failure Mode", style="yellow")
    table.add_column("Role", style="blue")

    max_length = 200

    for sample in samples:
        table.add_row(
            sample.id,
            sample.question[:max_length] + "..." if len(sample.question) > max_length else sample.question,
            sample.gold_answer[:max_length] + "..." if len(sample.gold_answer) > max_length else sample.gold_answer,
            str(sample.failure_mode.value) if hasattr(sample.failure_mode, "value") else str(sample.failure_mode),
            str(sample.role.value) if hasattr(sample.role, "value") else str(sample.role),
        )

    console.print(table)

In [14]:
show_qa_samples(test_samples[25:27])

                                                 Test Q&A Samples                                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ ID                                       ┃ Question         ┃ Answer          ┃ Failure Mode     ┃ Role         ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ test-cfpb_credit_card_agreement::p0029-1 │ Under the        │ If the borrower │ domain_vocabula… │ policy_dense │
│                                          │ Vermont Law      │ does not pay,   │                  │              │
│                                          │ Notice to        │ the lender has  │                  │              │
│                                          │ Co-signer, what  │ a legal right   │                  │              │
│                                          │ specific leg...  │ t...            │                  │              │
├──────────────────────────────────────────┼──────────────────┼─────────────────┼──────────────────┼──────────────┤
│ test-cfpb_credit_card_agreement::p0029-2 │ Under Vermont    │ Under Vermont   │ multi_constrain… │ policy_dense │
│                                          │ law, if a        │ law, the        │                  │              │
│                                          │ co-signer signs  │ co-signer is    │                  │              │
│                                          │ a loan note,     │ equally liable  │                  │              │
│                                          │ what is...       │ for the r...    │                  │              │
└──────────────────────────────────────────┴──────────────────┴─────────────────┴──────────────────┴──────────────┘

## 4. Baseline inference with the small model

Plug in your small model client here (local GGUF, Ollama, or HF 4-bit model).

In [15]:
small_model = create_small_model_client()

predictions = run_inference(small_model, test_samples)
console.print(f"[bold green]Collected:[/bold green] [cyan]{len(predictions)}[/cyan] baseline predictions")


sample = predictions[0]
sample_dict = sample.to_dict() if hasattr(sample, "to_dict") else dict(sample)
pretty_json = Syntax.from_json(sample_dict, indent=2) if hasattr(Syntax, "from_json") else None

if pretty_json is not None:
    console.print(Panel(pretty_json, title="First Prediction"))
else:
    # Fallback if Syntax.from_json is not available
    import json

    formatted = json.dumps(sample_dict, indent=2)
    console.print(Panel(formatted, title="First Prediction"))

2026-07-28 21:32:19,285 INFO aieng.syn_data.text.small_model: Creating small model client for model: qwen2.5:3b-instruct
2026-07-28 21:32:19,287 DEBUG urllib3.connectionpool: Starting new HTTP connection (1): 127.0.0.1:11434
2026-07-28 21:32:31,845 DEBUG urllib3.connectionpool: http://127.0.0.1:11434 "POST /v1/chat/completions HTTP/1.1" 200 498
2026-07-28 21:32:31,847 DEBUG urllib3.connectionpool: Starting new HTTP connection (1): 127.0.0.1:11434
2026-07-28 21:32:51,691 DEBUG urllib3.connectionpool: http://127.0.0.1:11434 "POST /v1/chat/completions HTTP/1.1" 200 768
2026-07-28 21:32:51,693 DEBUG urllib3.connectionpool: Starting new HTTP connection (1): 127.0.0.1:11434
2026-07-28 21:33:10,334 DEBUG urllib3.connectionpool: http://127.0.0.1:11434 "POST /v1/chat/completions HTTP/1.1" 200 761
2026-07-28 21:33:10,336 DEBUG urllib3.connectionpool: Starting new HTTP connection (1): 127.0.0.1:11434
2026-07-28 21:33:25,479 DEBUG urllib3.connectionpool: http://127.0.0.1:11434 "POST /v1/chat/compl

Collected: 60 baseline predictions

╭─────────────────────────────────────────────── First Prediction ────────────────────────────────────────────────╮
│ {                                                                                                               │
│   "id": "test-cfpb_credit_card_agreement::p0005-0",                                                             │
│   "question": "According to the policy, what are the two specific conditions under which new purchases posted   │
│ to an account during a billing cycle will not incur a finance charge for that cycle? Please provide your        │
│ response as a numbered list containing exactly two items.",                                                     │
│   "gold_answer": "1. You had a zero or credit balance at the beginning of that billing cycle.\n2. You paid the  │
│ entire new balance on the previous cycle's billing statement by the payment due date of that statement.",       │
│   "model_answer": "1. The beginning balance of the account at the start of the billing cycle is zero.\n2. The   │
│ entire new balance on the previous cycle's billing statement is paid by the payment due date of that            │
│ statement.",                                                                                                    │
│   "failure_mode": "format_non_compliance",                                                                      │
│   "doc_id": "cfpb_credit_card_agreement",                                                                       │
│   "para_id": "cfpb_credit_card_agreement::p0005"                                                                │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 5. LLM-as-judge baseline scores

In [ ]:
judge = create_judge_client()

baseline_scores = score_predictions(judge, test_samples, predictions)
baseline_summary = save_baseline_results(
    predictions,
    baseline_scores,
    test_samples,
    predictions_path=BASELINE_PREDICTIONS_PATH,
    scores_path=BASELINE_SCORES_PATH,
)

# TOOD: save the baseline score for one sample and print it here

2026-07-28 22:00:51,368 INFO aieng.syn_data.text.judge: Scoring model answer for sample: test-cfpb_credit_card_agreement::p0005-0 with model answer: 1. The beginning balance of the account at the start of the billing cycle is zero.
2. The entire new balance on the previous cycle's billing statement is paid by the payment due date of that statement.
2026-07-28 22:00:51,370 DEBUG urllib3.connectionpool: Starting new HTTPS connection (1): proxy.vectorinstitute.ai:443


2026-07-28 22:00:53,207 DEBUG urllib3.connectionpool: https://proxy.vectorinstitute.ai:443 "POST /v1/chat/completions HTTP/1.1" 200 None
2026-07-28 22:00:53,209 DEBUG aieng.syn_data.text.clients: *********** Extracted JSON payload: *********** 
{
    "correctness": 5,
    "coherence": 5,
    "instruction_following": 5,
    "factual_plausibility": 5,
    "reasoning": "The model answer is completely correct, matches the reference perfectly, and strictly follows the formatting instruction to provide a numbered list with exactly two items."
*********** End of JSON payload ***********
2026-07-28 22:00:53,211 INFO aieng.syn_data.text.judge: Scoring model answer for sample: test-cfpb_credit_card_agreement::p0005-1 with model answer: New purchases posted to your account during a billing cycle will not incur a finance charge if you meet one of the following specific conditions:

1. You had a zero or credit balance at the beginning of that billing cycle.
2. You paid the entire new balance on the

### Baseline score summary

In [17]:
# Create a table to display the baseline_summary
table = Table(title="Baseline Summary", highlight=True)

# Define the columns based on the baseline_summary structure
table.add_column("Failure Mode", style="bold cyan")
table.add_column("Correctness", justify="right", style="green")
table.add_column("Coherence", justify="right", style="green")
table.add_column("Instruction Following", justify="right", style="green")
table.add_column("Factual Plausibility", justify="right", style="green")
table.add_column("Average", justify="right", style="bold yellow")

# Add the "overall" scores as the first row
overall = baseline_summary.get("overall", {})
table.add_row(
    "[b]Overall[/b]",
    f"{overall.get('correctness', 0):.2f}",
    f"{overall.get('coherence', 0):.2f}",
    f"{overall.get('instruction_following', 0):.2f}",
    f"{overall.get('factual_plausibility', 0):.2f}",
    f"{overall.get('average', 0):.2f}",
)

# Add a row for each failure mode
by_failure = baseline_summary.get("by_failure_mode", {})
for mode, scores in by_failure.items():
    table.add_row(
        mode,
        f"{scores.get('correctness', 0):.2f}",
        f"{scores.get('coherence', 0):.2f}",
        f"{scores.get('instruction_following', 0):.2f}",
        f"{scores.get('factual_plausibility', 0):.2f}",
        f"{scores.get('average', 0):.2f}",
    )

console.print(table)

                                                Baseline Summary                                                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ Failure Mode              ┃ Correctness ┃ Coherence ┃ Instruction Following ┃ Factual Plausibility ┃ Average ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ Overall                   │        4.28 │      4.98 │                  4.63 │                 4.58 │    4.62 │
│ format_non_compliance     │        4.70 │      5.00 │                  4.75 │                 5.00 │    4.86 │
│ domain_vocabulary_drift   │        3.90 │      5.00 │                  5.00 │                 4.10 │    4.50 │
│ multi_constraint_collapse │        4.35 │      4.90 │                  5.00 │                 4.40 │    4.66 │
│ refusal_calibration       │        4.00 │      5.00 │                  4.15 │                 4.50 │    4.41 │
└───────────────────────────┴─────────────┴───────────┴───────────────────────┴──────────────────────┴─────────┘